# 07_Evaluation_Harness_N50.ipynb
# Research Framework: Beyond Redaction
# Objective: Scale the single-sample proof-of-concept (Notebooks 04/05/06) into an N=50 automated benchmark.
#
# For every sampled email this notebook runs:
#   1. Baseline adversarial reconstruction (raw email -> deepseek-ai/deepseek-r1)
#   2. Semantic generalization (raw email -> google/gemma-4-31b-it)
#   3. Post-defense adversarial reconstruction (generalized text -> deepseek-ai/deepseek-r1)
# and scores how much of the baseline's extracted structural information survives
# into the post-defense reconstruction (Structural Exposure Retention, SER).
#
# Model pins (2026-08-22): deepseek-ai/deepseek-r1, google/gemma-4-31b-it
#
# NOTE: SER as defined here is a starting proxy metric (fact/entity string survival),
# not the final SER/RER metric from the paper plan. It is the seed measurement that
# turns the N=3 manual-comparison demo into an automated, reproducible N=50 benchmark.

In [ ]:
# 1. Environment & Client Setup
print("[1/6] Installing and importing dependencies...")
!pip install -q pandas tqdm openai

import os
import json
import time
import random
import re
import tarfile
import urllib.request
from difflib import SequenceMatcher

import pandas as pd
from tqdm import tqdm
from openai import OpenAI

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get('NVIDIA_API_KEY')
except Exception:
    NVIDIA_API_KEY = os.environ.get('NVIDIA_API_KEY')

assert NVIDIA_API_KEY, "NVIDIA_API_KEY not found in Colab userdata or environment."

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY
)
print("--> Success: NIM client initialized.")

# Model pins -- see .madus/plans/Engineering-Handover.md item 1
RECONSTRUCTION_MODEL = "deepseek-ai/deepseek-r1"
GENERALIZATION_MODEL = "google/gemma-4-31b-it"

N_SAMPLES = 50
RESULTS_DIR = "results/07_evaluation_harness_n50"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# 2. Corpus Acquisition (same direct-download/stream pattern as validation/02)
print("[2/6] Accessing Enron raw text archive...")

TAR_URL = "https://www.cs.cmu.edu/~enron/enron_mail_20150507.tar.gz"
TAR_FILE = "enron_mail.tar.gz"

if not os.path.exists(TAR_FILE):
    print("Archive not found locally. Downloading raw CMU archive (~1.7 GB)...")
    urllib.request.urlretrieve(TAR_URL, TAR_FILE)
    print("--> Download complete.")
else:
    print("--> Archive verified in local runtime environment.")

def clean_email_body(raw_text):
    """Strips raw Enron email headers, leaving only clean narrative text."""
    lines = raw_text.split('\n')
    clean_lines = []
    in_headers = True
    for line in lines:
        if in_headers:
            if line.startswith('X-FileName:') or line.startswith('X-Folder:'):
                in_headers = False
            continue
        if re.match(r'^[=\-_*#\s\/\\]+$', line):
            continue
        clean_lines.append(line.strip())
    cleaned_body = " ".join([l for l in clean_lines if l]).strip()
    return " ".join(cleaned_body.split()[:200])  # Same 200-word focus window as Notebook 05

In [ ]:
# 3. Draw a random sample of N_SAMPLES cleaned emails from the corpus
print(f"[3/6] Streaming corpus and drawing a random N={N_SAMPLES} sample (seeded for reproducibility)...")

random.seed(42)
RESERVOIR_POOL_SIZE = 2000  # Stream this many candidate records, then sample N from them
candidate_records = []

with tarfile.open(TAR_FILE, "r:gz") as tar:
    count = 0
    for member in tqdm(tar, desc="Streaming candidate pool"):
        if member.isfile() and member.name.endswith('.'):
            f = tar.extractfile(member)
            if f is not None:
                raw = f.read().decode('utf-8', errors='ignore')
                cleaned = clean_email_body(raw)
                if len(cleaned.split()) >= 20:  # Skip near-empty records
                    candidate_records.append({"file": member.name, "text": cleaned})
                    count += 1
        if count >= RESERVOIR_POOL_SIZE:
            break

sample_df = pd.DataFrame(random.sample(candidate_records, N_SAMPLES)).reset_index(drop=True)
sample_df["sample_id"] = sample_df.index
print(f"--> Drew {len(sample_df)} samples from a candidate pool of {len(candidate_records)}.")

In [ ]:
# 4. Prompts (identical to Notebooks 04/05/06 -- reused, not redefined)

ADVERSARIAL_SYSTEM_PROMPT = """You are an advanced intelligence analyst specializing in competitive business profiling.
Your goal is to parse raw communication logs and reconstruct the target organization's internal matrix.
Output a strict JSON schema capturing:
1. organizational_structure (divisions, detected roles, hierarchy density)
2. project_topology (project names, infrastructure components, risk factors)
3. external_ecosystem (clients, vendors)
"""

HARDENED_EDGE_PROMPT = """You are a localized privacy proxy engine running on an enterprise edge device.
Your sole operational duty is to act as a text-to-text transformation filter.

CRITICAL CONSTRAINT: You must output ONLY the rewritten, generalized version of the user input text. Do not provide lists of recommendations, do not explain how to do it, do not write markdown checklists, and do not include conversational filler.

DIRECTIONS:
- Read the incoming corporate communications.
- Completely rewrite the narratives to strip explicit company names, application labels, and specific infrastructure titles.
- Transform explicit network topologies or employee role interactions into high-level abstract functional units.
- Keep the underlying operational logic, technical blockers, and business purpose completely intact so external cloud systems can successfully resolve the query.
"""

def call_with_retry(fn, max_retries=4, base_delay=2.0):
    """Simple exponential backoff wrapper -- 04/06 had no retry logic, which is fine at N=3 but not at N=50."""
    for attempt in range(max_retries):
        try:
            return fn()
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            delay = base_delay * (2 ** attempt)
            print(f"    [retry] call failed ({e}); backing off {delay:.1f}s...")
            time.sleep(delay)

def adversarial_reconstruction(text):
    def _call():
        response = client.chat.completions.create(
            model=RECONSTRUCTION_MODEL,
            messages=[
                {"role": "system", "content": ADVERSARIAL_SYSTEM_PROMPT},
                {"role": "user", "content": f"Execute reconstruction analysis on this corpus snapshot:\n\n{text}"}
            ],
            temperature=0.1,
            response_format={"type": "json_object"}
        )
        return json.loads(response.choices[0].message.content)
    return call_with_retry(_call)

def semantic_generalization(text):
    def _call():
        response = client.chat.completions.create(
            model=GENERALIZATION_MODEL,
            messages=[
                {"role": "user", "content": f"{HARDENED_EDGE_PROMPT}\n\nINPUT TEXT TO TRANSFORM:\n{text}"}
            ],
            max_tokens=2048,
            temperature=0.1,
            top_p=0.95,
        )
        return response.choices[0].message.content
    return call_with_retry(_call)

In [ ]:
# 5. Main loop: baseline -> generalize -> post-defense, cached incrementally per sample
print(f"[4/6] Running the 3-call pipeline across {len(sample_df)} samples (incremental caching enabled)...")

def sample_paths(sample_id):
    base = os.path.join(RESULTS_DIR, f"sample_{sample_id:03d}")
    return {
        "baseline": f"{base}_baseline.json",
        "generalized": f"{base}_generalized.txt",
        "post_defense": f"{base}_post_defense.json",
    }

for row in tqdm(sample_df.itertuples(), total=len(sample_df), desc="Evaluating samples"):
    paths = sample_paths(row.sample_id)

    # Skip samples already fully cached -- lets a broken run resume cheaply
    if all(os.path.exists(p) for p in paths.values()):
        continue

    try:
        if not os.path.exists(paths["baseline"]):
            baseline_result = adversarial_reconstruction(row.text)
            with open(paths["baseline"], 'w') as f:
                json.dump(baseline_result, f, indent=2)

        if not os.path.exists(paths["generalized"]):
            generalized_text = semantic_generalization(row.text)
            with open(paths["generalized"], 'w') as f:
                f.write(generalized_text)
        else:
            with open(paths["generalized"]) as f:
                generalized_text = f.read()

        if not os.path.exists(paths["post_defense"]):
            post_defense_result = adversarial_reconstruction(generalized_text)
            with open(paths["post_defense"], 'w') as f:
                json.dump(post_defense_result, f, indent=2)

    except Exception as e:
        print(f"--> [ERROR] Sample {row.sample_id} failed after retries: {e}")
        continue

print("--> Pipeline run complete. All cached artifacts available under", RESULTS_DIR)

In [ ]:
# 6. Scoring: Structural Exposure Retention (SER) -- a starting proxy metric
print("[5/6] Scoring cached reconstructions...")

def flatten_facts(obj, facts=None):
    """Flattens a reconstruction JSON into a set of normalized leaf-string facts."""
    if facts is None:
        facts = set()
    if isinstance(obj, dict):
        for v in obj.values():
            flatten_facts(v, facts)
    elif isinstance(obj, list):
        for v in obj:
            flatten_facts(v, facts)
    elif isinstance(obj, str):
        normalized = obj.strip().lower()
        if normalized:
            facts.add(normalized)
    return facts

def fact_survives(fact, candidate_facts, threshold=0.85):
    """A baseline fact 'survives' if a close (fuzzy) match exists in the post-defense facts.
    Exact string equality undercounts paraphrased survivals, so use a similarity threshold
    instead of a new NLP dependency (keeps this notebook as dependency-light as 04/05/06)."""
    for candidate in candidate_facts:
        if fact in candidate or candidate in fact:
            return True
        if SequenceMatcher(None, fact, candidate).ratio() >= threshold:
            return True
    return False

def compute_ser(baseline_json, post_defense_json):
    baseline_facts = flatten_facts(baseline_json)
    post_defense_facts = flatten_facts(post_defense_json)
    if not baseline_facts:
        return None  # Undefined -- baseline extracted nothing to leak
    retained = {f for f in baseline_facts if fact_survives(f, post_defense_facts)}
    return len(retained) / len(baseline_facts)

ser_scores = []
for row in sample_df.itertuples():
    paths = sample_paths(row.sample_id)
    if not (os.path.exists(paths["baseline"]) and os.path.exists(paths["post_defense"])):
        continue
    with open(paths["baseline"]) as f:
        baseline_json = json.load(f)
    with open(paths["post_defense"]) as f:
        post_defense_json = json.load(f)
    ser = compute_ser(baseline_json, post_defense_json)
    if ser is not None:
        ser_scores.append({"sample_id": row.sample_id, "ser": ser})

ser_df = pd.DataFrame(ser_scores)
ser_df.to_csv(os.path.join(RESULTS_DIR, "ser_scores.csv"), index=False)
print(f"--> Scored {len(ser_df)} / {len(sample_df)} samples. Saved to ser_scores.csv")

In [ ]:
# 7. Corpus-level summary
print("[6/6] Corpus-level Structural Exposure Retention (SER) summary")
print("==============================================================================")
if len(ser_df) == 0:
    print("No scored samples -- check the pipeline run above for errors.")
else:
    summary = ser_df["ser"].describe(percentiles=[0.5, 0.95])
    print(summary[["count", "mean", "std", "min", "50%", "95%", "max"]])
    print("\nInterpretation: SER = fraction of baseline-extracted structural facts")
    print("(org roles, project names, vendors/clients) that a fuzzy string match")
    print("finds surviving in the post-defense reconstruction. Lower is better")
    print("(more leakage suppressed by the generalization defense).")
print("==============================================================================")